In [29]:
import pandas as pd
import numpy as np

# Merging the Data

In [30]:
SteamReviewsData = pd.read_csv('Data1-steam.csv')
VideoGamesData = pd.read_csv('Data2-videoGames.csv')

# print(SteamReviewsData.to_string()) 
# print(VideoGamesData.to_string()) 

VideoGamesData["match_key"] = VideoGamesData["Name"].str.strip().str.lower()
SteamReviewsData["match_key"] = SteamReviewsData["name"].str.strip().str.lower()

ColumnsSteamReviewsData = SteamReviewsData[["match_key", "genres", "positive_ratings", "negative_ratings"]].copy()

ColumnsSteamReviewsData["steam_total_reviews"] = ColumnsSteamReviewsData["positive_ratings"] + ColumnsSteamReviewsData["negative_ratings"]

ColumnsSteamReviewsData["steam_positive_review_ratio"] = ColumnsSteamReviewsData["positive_ratings"] / ColumnsSteamReviewsData["steam_total_reviews"]

ColumnsSteamReviewsData.drop(columns=["positive_ratings", "negative_ratings"], inplace=True)

MergedData = pd.merge(VideoGamesData, ColumnsSteamReviewsData, how='left', on='match_key')

MergedData.drop(columns=["match_key"], inplace=True)
MergedData.to_csv('MergedData.csv', index=False)

# print(MergedData.to_string()) 

# Cleaning and Imputation of Missing Data

### Cleaning and preparing player reviews

In [31]:
#round the steam score to 1 decimal place and scale it up so its values are betwwen 0 and 10
MergedData['steam_score'] = (MergedData['steam_positive_review_ratio'] * 10).round(1)

Producting single player score
* Scenario 1: Only User_Score available -> use it 
* Scenario 2: Only steam_score available -> use it 
* Scenario 3: Both available -> get standard weighted average 
    * I found out which kind of average I need using Gemini AI with this prompt: I have a pandas .cvs file where some of my User_Count column is low in number (max ~8,715), while my steam_total_reviews column is extremely high in number (30k+ is common). I need to calculate a weighted average of User_Score and steam_score. What kind of average do i need?
        * no code was taken from  Gemini

In [33]:
#scenarios 1 and 2
rowHasSteamScore = MergedData['steam_score'].notna()
rowHasUserScore = MergedData['User_Score'].notna()

#scenario 3
MergedData['Weight_User_Count'] = np.log1p(MergedData['User_Count']) #e.g. 8000 becomes 9.0
MergedData['Weight_Steam_Reviews'] = np.log1p(MergedData['steam_total_reviews']) #e.g. 35000 becomes 10.5

MergedData['total_u_score_weight'] = MergedData['Weight_User_Count'] + MergedData['Weight_Steam_Reviews']

# avg
scenario3 = ((MergedData['User_Score'] * MergedData['Weight_User_Count']) +  (MergedData['steam_score'] * MergedData['Weight_Steam_Reviews'])) / MergedData['total_u_score_weight']


# Apply scenarios
conditions = [
    rowHasUserScore & ~rowHasSteamScore,   # Scenario 1 
    ~rowHasUserScore & rowHasSteamScore,   # Scenario 2 
    rowHasUserScore & rowHasSteamScore ]  # Scenario 3 

choices = [
    MergedData['User_Score'], # Scenario 1 
    MergedData['steam_score'], # Scenario 2
    scenario3] # Scenario 3

# MergedData['transformed_user_score'] = np.select(conditions, choices, 0)
MergedData['transformed_user_score'] = np.select(conditions, choices, default=np.nan)


MergedData['transformed_user_score'] = (MergedData['transformed_user_score']).round(1)
MergedData = MergedData.drop(columns=['Weight_User_Count', 'Weight_Steam_Reviews', 'total_u_score_weight'])


# remove rows which don't have a final score
rows_before_Uscore_clean = len(MergedData)
print(f"The number of rows is {rows_before_Uscore_clean} before removing rows without a final score.")

rows_before = len(MergedData)
MergedData = MergedData[MergedData['transformed_user_score'].notna()]

rows_after_Uscore_clean = len(MergedData)
print(f"Rows dropped (no player score): {rows_before - rows_after_Uscore_clean}")
print(f"Rows remaining: {rows_after_Uscore_clean}")

MergedData.to_csv('MergedData_clean.csv', index=False)

The number of rows is 8061 before removing rows without a final score.
Rows dropped (no player score): 0
Rows remaining: 8061


### Cleaning and preparing critic reviews